# Searching for Layer Dependencies

## Overview

This notebook will look for any items (web maps, apps) which are dependent upon a layer. This is a great tool to use prior to removing or replacing a layer, to ensure that you've addressed anywhere this layer shows up.

In brief, this notebook will:
1. Take a given service's ItemID
2. Find the associated service URL
3. Iterate over all maps in your org, looking for your service in
    1. Operational layers
    2. Basemap layers
    3. Anywhere else (search widget, etc.)
    
Matches will be printed out for user review.

## Setup

#### Run this cell to connect to your GIS and get started:

In [1]:
from arcgis.gis import GIS
import pandas as pd

gis = GIS("home")

You are logged on as isaiah.bennett@audubon.org_audubon with an administrator role, proceed with caution.


Run this cell to specify the service you're looking for by its `ItemID`.

Leave the first input blank to search by the service URL instead. Useful if you're looking for a specific sub-layer of the service.

In [2]:
find_id = "91a7e4ee709646febe991c74541ac2e7"
find_url = gis.content.get(find_id).url if find_id else input('Service URL: ')
print(find_url)

https://services1.arcgis.com/lDFzr3JyGEn5Eymu/arcgis/rest/services/Audubon_Assets_Public_View/FeatureServer


## The Search

### Web Maps
First, we'll be getting a list of **all** the web maps in your org.

For each web map, it's as simple as using `get_data()` to return the JSON definition of the map as a string. We can then use `find` together with our service URL to see if it shows up.

Our `map_list` is created using **list comprehension** together with a **ternary operator**, which are very handy features for keeping your Python concise.

#### Searching outside your org
Add the parameter `outside_org=True` to the **search** to look for maps from *any* AGOL organization that are shared publicly. Obviously, that's going to be a LOT of information to look through, so only do that if you're certain you want to, then go get some coffee or something while it executes.

In [3]:
webmaps = gis.content.search('', item_type='Web Map', max_items=-1, outside_org=False)

map_list = [m for m in webmaps if str(m.get_data()).find(find_url) > -1]

### Web Apps
Now we'll be getting a list of **all** the web apps in your org. There are different `item_type`s for each, so we need to perform multiple queries and merge them together. Again, list comprehension makes this a nice short line, rather than nested for-loops.

In [4]:
apptypes = ['Application', 'Dashboard', 'Story Map', 'Web Experience']

webapps = [item for sublist in [gis.content.search('', item_type=t, max_items=-1, outside_org=False) for t in apptypes] for item in sublist]

Now that we have our aps, searching through them is a little different. A web map may reference your layer directly, using its URL **or** its ItemID. It may also simply reference the web map which contains your layer.

To search each of these, we create a list of `criteria`, then we use `any` to see if one or more of the criteria evaluate to `True`. When that is the case, we append that web app to a list of apps.

In [5]:
app_list = []

for w in webapps:
    
    try:
        wdata = str(w.get_data())

        criteria = [
            wdata.find(find_url) > -1,
            wdata.find(find_id) > -1,
            any([wdata.find(m.id) > -1 for m in map_list])
        ]
        
        if any(criteria):
            app_list.append(w)
    
    # Some apps don't have data, so we'll just skip them if they throw a TypeError
    except:
        continue

## Reviewing the Output

Now we've got our lists of maps and apps. But to get them in a user-friendly format, we'll use `pandas` to cobble together a dataframe.

In [6]:
def shared_info(item):
    """
    Returns structured sharing info for an AGO Item.
    """
    sw = item.shared_with
    return {
        'shared_with_org': sw['org'],
        'shared_with_everyone': sw['everyone'],
        'shared_with_groups': sw['groups']  # list of group IDs
    }


dependencies = pd.concat(
    [
        pd.DataFrame([
            {
                'title': a.title,
                'id': a.id,
                'type': a.type,
                'owner': a.owner,
                'created': pd.to_datetime(a.created, unit='ms'),
                'modified': pd.to_datetime(a.modified, unit='ms'),
                'views': a.numViews,
                'shared_with_org': shared_info(a)['shared_with_org'],
                'shared_with_everyone': shared_info(a)['shared_with_everyone'],
                'shared_with_groups': shared_info(a)['shared_with_groups'],
                'url': f'{gis.url}/home/item.html?id={a.id}'#,
                #'data': a.get_data()
            }
            for a in app_list
        ]),
        
        pd.DataFrame([
            {
                'title': m.title,
                'id': m.id,
                'type': m.type,
                'owner': m.owner,
                'created': pd.to_datetime(m.created, unit='ms'),
                'modified': pd.to_datetime(m.modified, unit='ms'),
                'views': m.numViews,
                'shared_with_org': shared_info(m)['shared_with_org'],
                'shared_with_everyone': shared_info(m)['shared_with_everyone'],
                'shared_with_groups': shared_info(m)['shared_with_groups'],
                'url': f'{gis.url}/home/item.html?id={m.id}'#,
                #'data': m.get_data()
            }
            for m in map_list
        ])
    ]
)

dependencies.reset_index(inplace=True, drop=True)


dependencies.head()

/tmp/ipykernel_422/280657297.py:5: DeprecatedWarning: shared_with is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  sw = item.shared_with
/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_422/280657297.py:5: DeprecatedWarning: shared_with is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  sw = item.shared_with
/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_422/280657297.p

,title,id,type,owner,created,modified,views,shared_with_org,shared_with_everyone,shared_with_groups,url
0,Audubon Chapter Boundary Update Swipe Comparis...,f052f82d6c5d47499f2e85e65cb7af71,Web Mapping Application,Audubon_GIS,2020-12-03 19:11:26,2020-12-03 19:29:48,114,True,True,[],https://www.arcgis.com/home/item.html?id=f052f...
1,Upper Mississippi River Prioritization Viewer,9688db3a807c48c8ab77605c92e3f122,Web Mapping Application,GIS_Help_audubon,2022-12-05 17:08:53,2024-03-26 16:58:21,619,True,True,[],https://www.arcgis.com/home/item.html?id=9688d...
2,Center&Sanc Layers Filter Data,42ed9dba46ef44e7a71e163a1de21e64,Web Mapping Application,GIS_Help_audubon,2022-01-06 18:48:20,2022-05-13 15:19:27,29,False,False,[],https://www.arcgis.com/home/item.html?id=42ed9...
3,Audubon_homepage,f1593652a40c438b81e774159ab4fd17,Web Mapping Application,GIS_Help_audubon,2015-01-23 20:15:21,2019-08-22 16:54:55,142,True,True,[],https://www.arcgis.com/home/item.html?id=f1593...
4,Bass Pro Shops and Cabela's,e26dc310492f45598e9aafeb58edd3fc,Web Mapping Application,louise.haupert@audubon.org_audubon,2025-04-23 13:26:54,2025-04-28 19:11:21,221,True,True,[],https://www.arcgis.com/home/item.html?id=e26dc...


In [7]:
# Write DataFrame to a temporary CSV
csv_path = "audubon_dependencies.csv"
dependencies.to_csv(csv_path, index=False)

# Upload CSV to AGO
item_props = {
    "type": "CSV",
    "title": "Audubon Assets Public View Layer Dependencies",
    "tags": "dependencies, layer usage"
}

uploaded_dependencies = gis.content.add(
    item_properties=item_props,
    data=csv_path
)

print("Uploaded:", uploaded_dependencies)

/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3579: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Uploaded: <Item title:"Audubon Assets Public View Layer Dependencies" type:CSV owner:isaiah.bennett@audubon.org_audubon>


## Looking at A Specific Item

In [8]:
dependencies.loc[3,'data']

KeyError: 'data'

## What now?

Now you've got your output, but what do you do with it? Well, you could use `dependencies.to_csv()` to save the dataframe to a file.
If you're just tracking down references to your service to replace or remove them, you can use the `url` column to go straight to the item's page in your portal and make the necessary adjustments.

## Taking it Further

Suppose you had *multiple* layers you wanted to run this for. You could, if you wanted, bundle all of this into a custom function that parses a list of IDs / URLs as its input. Or you could adjust the list of app types being searched for to get more refined results. I'll leave that to you!